# 🤖 Model Training
## Marketing Intelligence AI Platform

**Purpose:** Train, evaluate, and persist all four ML models.

**Sections:**
1. Environment Setup
2. Load Feature Store & Splits
3. Revenue Drop Risk — XGBoost + LightGBM Ensemble
4. Anomaly Detection — Isolation Forest
5. Customer Segmentation — K-Means
6. Creative Performance — CatBoost
7. SHAP Explainability (Revenue Risk)
8. Model Comparison Summary
9. Save All Artefacts

---
## 1. Environment Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import joblib

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Gradient boosting libraries
import xgboost as xgb
import lightgbm as lgb

# Explainability
import shap
shap.initjs()

# Project
from shared.data_loader import DataLoader
from shared.constants import (
    REVENUE_COL, SPEND_COL, FEATURE_STORE,
    XGBOOST_MODEL, LIGHTGBM_MODEL, SHAP_EXPLAINER,
    ISOLATION_FOREST_MODEL, KMEANS_MODEL, CATBOOST_MODEL,
    FEATURE_COLUMNS_PATH
)
from settings import XGBOOST_PARAMS, LIGHTGBM_PARAMS, ISOLATION_FOREST_PARAMS, KMEANS_PARAMS

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

print('✅ Environment ready.')

---
## 2. Load Feature Store & Splits

In [ ]:
# TODO: Replace with: df = DataLoader.load_feature_store()
# and feature_cols = joblib.load(FEATURE_COLUMNS_PATH)

np.random.seed(42)
N = 2000

feature_cols = [
    'impressions','clicks','spend','conversions',
    'ctr','cpc','roas','cpa','rpc',
    'month','day_of_week','week_of_year','is_weekend',
    'revenue_roll7_mean','spend_roll7_mean','roas_roll7_mean',
]

X = np.random.randn(N, len(feature_cols))
y_binary   = (np.random.rand(N) > 0.7).astype(int)   # revenue drop flag
y_revenue  = np.random.uniform(0, 25000, N)            # continuous revenue

X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_revenue, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

---
## 3. Revenue Drop Risk — XGBoost + LightGBM Ensemble

In [ ]:
# --- XGBoost ---
xgb_model = xgb.XGBClassifier(**XGBOOST_PARAMS, eval_metric='logloss', use_label_encoder=False)

# TODO: Add early stopping: eval_set=[(X_val, y_val)], early_stopping_rounds=20
xgb_model.fit(X_train, y_train)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_preds = xgb_model.predict(X_test)
xgb_auc   = roc_auc_score(y_test, xgb_proba)

print(f'XGBoost  AUC-ROC: {xgb_auc:.4f}')
print(classification_report(y_test, xgb_preds, target_names=['No Drop','Revenue Drop']))

In [ ]:
# --- LightGBM ---
lgb_model = lgb.LGBMClassifier(**LIGHTGBM_PARAMS)

# TODO: Add callbacks=[lgb.early_stopping(20)] and eval_set
lgb_model.fit(X_train, y_train)

lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
lgb_preds = lgb_model.predict(X_test)
lgb_auc   = roc_auc_score(y_test, lgb_proba)

print(f'LightGBM AUC-ROC: {lgb_auc:.4f}')
print(classification_report(y_test, lgb_preds, target_names=['No Drop','Revenue Drop']))

In [ ]:
# --- Soft-voting Ensemble ---
ensemble_proba = (xgb_proba + lgb_proba) / 2
ensemble_preds = (ensemble_proba >= 0.5).astype(int)
ensemble_auc   = roc_auc_score(y_test, ensemble_proba)

print(f'Ensemble AUC-ROC: {ensemble_auc:.4f}')
print(classification_report(y_test, ensemble_preds, target_names=['No Drop','Revenue Drop']))

# Confusion matrix
cm = confusion_matrix(y_test, ensemble_preds)
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                title='Ensemble — Confusion Matrix',
                labels={'x':'Predicted','y':'Actual'},
                x=['No Drop','Drop'], y=['No Drop','Drop'])
fig.show()

---
## 4. Anomaly Detection — Isolation Forest

In [ ]:
iso_forest = IsolationForest(**ISOLATION_FOREST_PARAMS)
iso_forest.fit(X_train)

anomaly_scores = iso_forest.decision_function(X_test)
anomaly_labels = iso_forest.predict(X_test)  # +1 normal, -1 anomaly
n_anomalies    = (anomaly_labels == -1).sum()

print(f'Anomalies detected in test set: {n_anomalies} / {len(X_test)} ({n_anomalies/len(X_test)*100:.1f}%)')

# Score distribution
fig = px.histogram(x=anomaly_scores, nbins=50, title='Anomaly Score Distribution',
                   labels={'x': 'Decision Function Score'},
                   color_discrete_sequence=['steelblue'])
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Threshold')
fig.show()

# TODO: Evaluate against labelled anomaly ground truth if available.

---
## 5. Customer Segmentation — K-Means

In [ ]:
# Elbow method to find optimal K
inertias = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias[k] = km.inertia_

fig = px.line(x=list(inertias.keys()), y=list(inertias.values()),
              markers=True, title='Elbow Method — Inertia vs K',
              labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'})
fig.show()

In [ ]:
# Train final K-Means with chosen K
optimal_k = 5  # TODO: Update based on Elbow / Silhouette analysis.

kmeans = KMeans(n_clusters=optimal_k, **KMEANS_PARAMS)
labels = kmeans.fit_predict(X)

sil_score = silhouette_score(X, labels)
print(f'Silhouette Score (K={optimal_k}): {sil_score:.4f}')

# Segment size distribution
counts = pd.Series(labels).value_counts().sort_index()
fig = px.bar(x=counts.index.astype(str), y=counts.values,
             title=f'Segment Sizes (K={optimal_k})',
             labels={'x': 'Segment', 'y': 'Count'},
             color=counts.values, color_continuous_scale='Viridis')
fig.show()

---
## 6. Creative Performance — CatBoost

In [ ]:
try:
    from catboost import CatBoostRegressor
    from settings import CATBOOST_PARAMS

    cb_model = CatBoostRegressor(**CATBOOST_PARAMS)

    # TODO: Pass cat_features list for categorical column indices.
    cb_model.fit(X_train_r, y_train_r, eval_set=(X_test_r, y_test_r), use_best_model=True)

    cb_preds = cb_model.predict(X_test_r)
    cb_mae   = mean_absolute_error(y_test_r, cb_preds)
    cb_rmse  = np.sqrt(mean_squared_error(y_test_r, cb_preds))
    cb_r2    = r2_score(y_test_r, cb_preds)

    print(f'CatBoost MAE  : {cb_mae:.2f}')
    print(f'CatBoost RMSE : {cb_rmse:.2f}')
    print(f'CatBoost R²   : {cb_r2:.4f}')

except ImportError:
    print('⚠️  catboost not installed. Run: pip install catboost')

---
## 7. SHAP Explainability (Revenue Risk — XGBoost)

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test[:200])

shap.summary_plot(
    shap_values, X_test[:200],
    feature_names=feature_cols,
    plot_type='bar',
    show=True
)

In [ ]:
# Beeswarm plot — full feature impact
shap.summary_plot(shap_values, X_test[:200], feature_names=feature_cols)

---
## 8. Model Comparison Summary

In [ ]:
summary = pd.DataFrame([
    {'Module': 'Revenue Risk', 'Model': 'XGBoost',     'Metric': 'AUC-ROC', 'Value': xgb_auc},
    {'Module': 'Revenue Risk', 'Model': 'LightGBM',    'Metric': 'AUC-ROC', 'Value': lgb_auc},
    {'Module': 'Revenue Risk', 'Model': 'Ensemble',    'Metric': 'AUC-ROC', 'Value': ensemble_auc},
    {'Module': 'Segmentation', 'Model': 'KMeans',      'Metric': 'Silhouette', 'Value': sil_score},
])

fig = px.bar(
    summary, x='Model', y='Value', color='Module',
    barmode='group', text_auto='.3f',
    title='Model Performance Comparison',
)
fig.show()
summary

---
## 9. Save All Artefacts

In [ ]:
import os

artefacts = [
    (xgb_model,   f'../{XGBOOST_MODEL}'),
    (lgb_model,   f'../{LIGHTGBM_MODEL}'),
    (explainer,   f'../{SHAP_EXPLAINER}'),
    (iso_forest,  f'../{ISOLATION_FOREST_MODEL}'),
    (kmeans,      f'../{KMEANS_MODEL}'),
]

for obj, path in artefacts:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    joblib.dump(obj, path)
    size_kb = os.path.getsize(path) / 1024
    print(f'✅ Saved: {path}  ({size_kb:.1f} KB)')

# CatBoost uses its own save format
# TODO: cb_model.save_model(f'../{CATBOOST_MODEL}')
# print(f'✅ Saved: ../{CATBOOST_MODEL}')